# DocMind Research v1 — FINAL Generation Evaluation

This notebook is **final-only**. It does not rerun DEV model selection.

## Before running
1. In Kaggle, enable a **GPU**.
2. Attach the dataset containing:
   - `docmind_generation_test_frozen.json`
   - `docmind_generation_noanswer_frozen.json`
3. **Restart the Kaggle session/kernel once**, then run this notebook from the top.

The first code cell fixes the Python package stack **before Transformers is imported**, including `bitsandbytes`. This prevents the cached "bitsandbytes unavailable" error from the previous notebook.

Expected final outputs:
- `qwen3_4b__test_generations.json`
- `qwen3_4b__noanswer_generations.json`
- `test_generation_results.csv`
- `test_generation_summary.csv`
- `noanswer_evaluation.csv`


In [1]:
# ============================================================
# CELL 1 — DEPENDENCY PREFLIGHT
# IMPORTANT: transformers/bitsandbytes are NOT imported here.
# ============================================================

import sys
import subprocess
import importlib.metadata as md

PINS = {
    "transformers": "4.57.1",
    "tokenizers": "0.22.1",
    "huggingface-hub": "0.36.0",
    "bitsandbytes": "0.50.0",
    "sentence-transformers": "5.4.1",
}

def version_of(package):
    try:
        return md.version(package)
    except md.PackageNotFoundError:
        return None

before = {pkg: version_of(pkg) for pkg in PINS}

print("Package versions before preflight:")
for pkg, ver in before.items():
    print(f"  {pkg:<23} {ver}")

to_install = [
    f"{pkg}=={wanted}"
    for pkg, wanted in PINS.items()
    if before[pkg] != wanted
]

# accelerate must exist before Transformers is imported because
# device_map='auto' and 4-bit loading use it.
try:
    md.version("accelerate")
    accelerate_missing = False
except md.PackageNotFoundError:
    accelerate_missing = True

if to_install:
    print("\nInstalling benchmark-compatible packages BEFORE importing Transformers...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-deps", "--force-reinstall",
        *to_install,
    ])

if accelerate_missing:
    print("Installing accelerate...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-deps", "accelerate",
    ])

after = {pkg: version_of(pkg) for pkg in PINS}

print("\nPackage versions after preflight:")
for pkg, ver in after.items():
    print(f"  {pkg:<23} {ver}")

bad = {
    pkg: (after[pkg], wanted)
    for pkg, wanted in PINS.items()
    if after[pkg] != wanted
}

if bad:
    raise RuntimeError(f"Package preflight failed: {bad}")

print("\n✅ Dependency preflight complete.")
print("✅ bitsandbytes was installed before Transformers import.")


Package versions before preflight:
  transformers            4.57.1
  tokenizers              0.22.1
  huggingface-hub         0.36.0
  bitsandbytes            0.50.0
  sentence-transformers   5.1.2

Installing benchmark-compatible packages BEFORE importing Transformers...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 22.8 MB/s eta 0:00:00

Package versions after preflight:
  transformers            4.57.1
  tokenizers              0.22.1
  huggingface-hub         0.36.0
  bitsandbytes            0.50.0
  sentence-transformers   5.4.1

✅ Dependency preflight complete.
✅ bitsandbytes was installed before Transformers import.


In [2]:
# ============================================================
# CELL 2 — IMPORTS + GPU CHECK
# ============================================================

from pathlib import Path
import json
import re
import gc
import time
import subprocess
import importlib.util

import numpy as np
import pandas as pd
import torch

# Import bitsandbytes before Transformers so availability is unambiguous.
import bitsandbytes as bnb
import transformers
import huggingface_hub

print("Transformers      :", transformers.__version__)
print("Hugging Face Hub  :", huggingface_hub.__version__)
print("bitsandbytes      :", bnb.__version__)
print("PyTorch           :", torch.__version__)
print("CUDA available    :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is OFF. Enable a Kaggle GPU accelerator, restart, and run from the top."
    )

print("GPU               :", torch.cuda.get_device_name(0))

FINAL_OUTPUT_DIR = Path("/kaggle/working/docmind_final_v1")
FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory  :", FINAL_OUTPUT_DIR)


Transformers      : 4.57.1
Hugging Face Hub  : 0.36.0
bitsandbytes      : 0.50.0
PyTorch           : 2.10.0+cu128
CUDA available    : True
GPU               : Tesla T4
Output directory  : /kaggle/working/docmind_final_v1


In [3]:
# ============================================================
# CELL 3 — LOAD FROZEN TEST + NO-ANSWER INPUTS
# ============================================================

KAGGLE_INPUT = Path("/kaggle/input")

def find_input_file(filename):
    matches = list(KAGGLE_INPUT.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Missing required input: {filename}\n"
            "Attach the Kaggle dataset containing this file."
        )

    if len(matches) > 1:
        print(f"⚠️ Multiple matches for {filename}; using:")
        for p in matches:
            print("  -", p)

    print(f"✅ {filename}\n   → {matches[0]}")
    return matches[0]

TEST_PATH = find_input_file("docmind_generation_test_frozen.json")
NOANSWER_PATH = find_input_file("docmind_generation_noanswer_frozen.json")

with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_payload = json.load(f)

with open(NOANSWER_PATH, "r", encoding="utf-8") as f:
    noanswer_payload = json.load(f)

FINAL_TEST_RECORDS = (
    test_payload["records"]
    if isinstance(test_payload, dict)
    else test_payload
)

FINAL_NOANSWER_RECORDS = (
    noanswer_payload["records"]
    if isinstance(noanswer_payload, dict)
    else noanswer_payload
)

assert len(FINAL_TEST_RECORDS) == 12, (
    f"Expected 12 TEST records, got {len(FINAL_TEST_RECORDS)}"
)

assert len(FINAL_NOANSWER_RECORDS) == 4, (
    f"Expected 4 no-answer records, got {len(FINAL_NOANSWER_RECORDS)}"
)

for record in FINAL_TEST_RECORDS + FINAL_NOANSWER_RECORDS:
    assert len(record.get("contexts", [])) == 5, (
        f"{record.get('question_id')} does not contain exactly 5 frozen contexts"
    )

print("\n✅ TEST records      :", len(FINAL_TEST_RECORDS))
print("✅ NO-ANSWER records :", len(FINAL_NOANSWER_RECORDS))
print("✅ 5 frozen contexts per question")


✅ docmind_generation_test_frozen.json
   → /kaggle/input/datasets/ripperdzz/dev-frozen/docmind_generation_test_frozen.json
✅ docmind_generation_noanswer_frozen.json
   → /kaggle/input/datasets/ripperdzz/dev-frozen/docmind_generation_noanswer_frozen.json

✅ TEST records      : 12
✅ NO-ANSWER records : 4
✅ 5 frozen contexts per question


In [4]:
# ============================================================
# CELL 4 — FROZEN DOCMIND PROMPT
# ============================================================

SYSTEM_PROMPT = """
You are DocMind, a grounded document question-answering system.

Rules:
1. Answer using ONLY the provided document sources.
2. Do not use outside knowledge.
3. If the answer cannot be determined from the sources, say exactly:
   "Not found in the provided documents."
4. Answer in the same language as the user's question.
5. Be concise but complete.
6. Cite supporting sources inline using [S1], [S2], etc.
7. Cite only sources that directly support the answer.
8. Never invent a citation.
9. Do not repeat the answer.
10. Do not add a separate references or citations section.
""".strip()

def build_user_prompt(record):
    blocks = []

    for ctx in record["contexts"]:
        blocks.append(
            f"""[{ctx['source_id']}]
Document: {ctx['document_id']}
Location: {ctx['location_type']} {ctx['location_value']}
Text:
{ctx['text']}""".strip()
        )

    sources = "\n\n".join(blocks)

    return f"""SOURCES

{sources}

QUESTION

{record['question']}

Answer using only the SOURCES.""".strip()

print("✅ Frozen prompt restored")


✅ Frozen prompt restored


In [5]:
# ============================================================
# CELL 5 — LOAD QWEN3-4B IN THE SAME 4-BIT NF4 MODE AS DEV
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

QUANT_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

gc.collect()
torch.cuda.empty_cache()

print("Loading:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=QUANT_CONFIG,
    device_map="auto",
    dtype=torch.float16,
)

model.eval()

is_4bit = bool(getattr(model, "is_loaded_in_4bit", False))
print("\n✅ Qwen3-4B loaded")
print("4-bit             :", is_4bit)

if not is_4bit:
    raise RuntimeError("Model did not load in 4-bit mode; stop the benchmark.")

try:
    print(
        "Model footprint    :",
        round(model.get_memory_footprint() / 1024**3, 2),
        "GB",
    )
except Exception:
    pass


Loading: Qwen/Qwen3-4B-Instruct-2507


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]


✅ Qwen3-4B loaded
4-bit             : True
Model footprint    : 2.42 GB


In [6]:
# ============================================================
# CELL 6 — GENERATION FUNCTION
# ============================================================

def generate_answer(tokenizer, model, record, max_new_tokens=180):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(record)},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    # Qwen is entirely on GPU in this 4-bit benchmark.
    input_device = next(model.parameters()).device
    inputs = {
        key: value.to(input_device)
        for key, value in inputs.items()
    }

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    input_len = inputs["input_ids"].shape[-1]
    generated_ids = output[0][input_len:]
    generated_tokens = int(generated_ids.shape[-1])

    answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    return {
        "answer": answer,
        "latency_seconds": float(elapsed),
        "generated_tokens": generated_tokens,
        "tokens_per_second": (
            float(generated_tokens / elapsed)
            if elapsed > 0
            else None
        ),
    }

print("✅ Generation function ready")


✅ Generation function ready


In [7]:
# ============================================================
# CELL 7 — RESUMABLE FINAL RUNNER + RUN 12 TEST / 4 NO-ANSWER
# ============================================================

from tqdm.auto import tqdm

def run_final_split(split_name, records, filename, max_new_tokens=180):
    save_path = FINAL_OUTPUT_DIR / filename

    # Resume safely if a previous partial result exists.
    existing = []

    if save_path.exists():
        try:
            with open(save_path, "r", encoding="utf-8") as f:
                existing = json.load(f)

            if not isinstance(existing, list):
                existing = []
        except Exception:
            existing = []

    completed_ids = {
        row["question_id"]
        for row in existing
        if "question_id" in row
    }

    results = list(existing)

    pending = [
        record
        for record in records
        if record["question_id"] not in completed_ids
    ]

    print(
        f"{split_name}: "
        f"{len(results)} already saved, "
        f"{len(pending)} remaining"
    )

    for record in tqdm(
        pending,
        desc=f"Qwen3-4B — {split_name}",
    ):
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        result = generate_answer(
            tokenizer=tokenizer,
            model=model,
            record=record,
            max_new_tokens=max_new_tokens,
        )

        peak_vram_gb = (
            torch.cuda.max_memory_allocated() / 1024**3
            if torch.cuda.is_available()
            else None
        )

        row = {
            "split": split_name,
            "question_id": record["question_id"],
            "question": record["question"],
            "language": record.get("language"),
            "question_type": record.get("question_type"),
            "answerable": record.get("answerable"),
            "expected_answer": record.get("expected_answer"),
            "answer": result["answer"],
            "latency_seconds": result["latency_seconds"],
            "generated_tokens": result["generated_tokens"],
            "tokens_per_second": result["tokens_per_second"],
            "peak_vram_gb": peak_vram_gb,
        }

        results.append(row)

        # Save after every question.
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(
                results,
                f,
                ensure_ascii=False,
                indent=2,
            )

    print(f"✅ Saved: {save_path}")
    return results


FINAL_TEST_GENERATIONS = run_final_split(
    split_name="TEST",
    records=FINAL_TEST_RECORDS,
    filename="qwen3_4b__test_generations.json",
    max_new_tokens=180,
)

FINAL_NOANSWER_GENERATIONS = run_final_split(
    split_name="NOANSWER",
    records=FINAL_NOANSWER_RECORDS,
    filename="qwen3_4b__noanswer_generations.json",
    max_new_tokens=180,
)

assert len(FINAL_TEST_GENERATIONS) == 12
assert len(FINAL_NOANSWER_GENERATIONS) == 4

print("\n🎉 Generation complete: 12 TEST + 4 no-answer")


TEST: 0 already saved, 12 remaining


Qwen3-4B — TEST:   0%|          | 0/12 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ Saved: /kaggle/working/docmind_final_v1/qwen3_4b__test_generations.json
NOANSWER: 0 already saved, 4 remaining


Qwen3-4B — NOANSWER:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Saved: /kaggle/working/docmind_final_v1/qwen3_4b__noanswer_generations.json

🎉 Generation complete: 12 TEST + 4 no-answer


In [8]:
# ============================================================
# CELL 8 — RUNTIME SUMMARY
# ============================================================

test_df = pd.DataFrame(FINAL_TEST_GENERATIONS)

runtime_summary = pd.DataFrame([{
    "questions": len(test_df),
    "avg_latency_seconds": float(test_df["latency_seconds"].mean()),
    "median_latency_seconds": float(test_df["latency_seconds"].median()),
    "avg_tokens_per_second": float(test_df["tokens_per_second"].mean()),
    "peak_vram_gb": float(test_df["peak_vram_gb"].max()),
    "avg_generated_tokens": float(test_df["generated_tokens"].mean()),
}])

runtime_summary.to_csv(
    FINAL_OUTPUT_DIR / "test_generation_runtime_summary.csv",
    index=False,
)

display(runtime_summary)
print("✅ Runtime summary saved")


,questions,avg_latency_seconds,median_latency_seconds,avg_tokens_per_second,peak_vram_gb,avg_generated_tokens
0,12,7.784247,7.187712,8.572914,1.537403,67.916667


✅ Runtime summary saved


In [9]:
# ============================================================
# CELL 9 — SAME AUTOMATIC QUALITY PROXY USED ON DEV
# This is a comparison heuristic, NOT factual accuracy.
# ============================================================

# Qwen is no longer needed. Free its GPU memory before loading
# the sentence embedding evaluator.
del model
gc.collect()
torch.cuda.empty_cache()

from sentence_transformers import SentenceTransformer

if importlib.util.find_spec("langdetect") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "langdetect",
    ])

from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0

EVAL_MODEL_NAME = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

evaluator = SentenceTransformer(
    EVAL_MODEL_NAME,
    device="cuda",
)

CITATION_RE = re.compile(r"\[S(\d+)\]")

def clean_eval_text(x):
    return str(x if x is not None else "").strip()

def split_sentences(text):
    return [
        x.strip()
        for x in re.split(
            r"(?<=[.!?؟])\s+|\n+",
            clean_eval_text(text),
        )
        if len(x.strip()) >= 3
    ]

def split_gold(text):
    return [
        x.strip()
        for x in re.split(
            r"[.;!?؟]\s*|\n+",
            clean_eval_text(text),
        )
        if len(x.strip()) >= 3
    ]

def lang_safe(text):
    try:
        return detect(clean_eval_text(text))
    except Exception:
        return "unknown"

record_lookup = {
    r["question_id"]: r
    for r in FINAL_TEST_RECORDS
}

all_texts = set()

for row in FINAL_TEST_GENERATIONS:
    qid = row["question_id"]
    answer = clean_eval_text(row["answer"])
    gold = clean_eval_text(row["expected_answer"])

    all_texts.update([answer, gold])
    all_texts.update(split_sentences(answer))
    all_texts.update(split_gold(gold))

    for ctx in record_lookup[qid]["contexts"]:
        all_texts.add(
            clean_eval_text(ctx["text"])
        )

all_texts.discard("")
all_texts = list(all_texts)

embs = evaluator.encode(
    all_texts,
    batch_size=64,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
)

EMB = {
    text: emb
    for text, emb in zip(all_texts, embs)
}

def sim(a, b):
    return float(
        np.clip(
            np.dot(EMB[a], EMB[b]),
            0.0,
            1.0,
        )
    )

rows = []

for generation in FINAL_TEST_GENERATIONS:
    qid = generation["question_id"]
    frozen = record_lookup[qid]

    answer = clean_eval_text(generation["answer"])
    gold = clean_eval_text(generation["expected_answer"])
    contexts = [
        clean_eval_text(c["text"])
        for c in frozen["contexts"]
    ]

    correctness = sim(answer, gold)

    units = split_gold(gold)
    coverage = (
        float(np.mean([sim(unit, answer) for unit in units]))
        if units
        else correctness
    )

    sentences = split_sentences(answer)
    groundedness = (
        float(
            np.mean([
                max(sim(sentence, ctx) for ctx in contexts)
                for sentence in sentences
            ])
        )
        if sentences
        else 0.0
    )

    citations = [
        int(x)
        for x in CITATION_RE.findall(answer)
    ]

    valid = [
        c
        for c in citations
        if 1 <= c <= len(contexts)
    ]

    citation_presence = float(bool(citations))

    citation_validity = (
        len(valid) / len(citations)
        if citations
        else 0.0
    )

    citation_support = (
        float(
            np.mean([
                sim(answer, contexts[c - 1])
                for c in valid
            ])
        )
        if valid
        else 0.0
    )

    expected_lang = generation.get("language")
    detected_lang = lang_safe(answer)

    language_match = float(
        detected_lang == expected_lang
    )

    automatic_quality = (
        0.35 * correctness
        + 0.20 * coverage
        + 0.20 * groundedness
        + 0.10 * citation_validity
        + 0.10 * citation_support
        + 0.05 * language_match
    )

    rows.append({
        "question_id": qid,
        "language": expected_lang,
        "question_type": generation.get("question_type"),
        "semantic_correctness": correctness,
        "gold_coverage": coverage,
        "groundedness": groundedness,
        "citation_presence": citation_presence,
        "citation_validity": citation_validity,
        "citation_support": citation_support,
        "language_compliance": language_match,
        "latency_seconds": generation["latency_seconds"],
        "tokens_per_second": generation["tokens_per_second"],
        "peak_vram_gb": generation["peak_vram_gb"],
        "automatic_quality_score": automatic_quality,
    })


TEST_GENERATION_DF = pd.DataFrame(rows)

TEST_GENERATION_SUMMARY = TEST_GENERATION_DF.agg({
    "semantic_correctness": "mean",
    "gold_coverage": "mean",
    "groundedness": "mean",
    "citation_presence": "mean",
    "citation_validity": "mean",
    "citation_support": "mean",
    "language_compliance": "mean",
    "latency_seconds": "mean",
    "tokens_per_second": "mean",
    "peak_vram_gb": "max",
    "automatic_quality_score": "mean",
}).to_frame().T

TEST_GENERATION_SUMMARY.insert(
    0,
    "questions",
    len(TEST_GENERATION_DF),
)

TEST_GENERATION_DF.to_csv(
    FINAL_OUTPUT_DIR / "test_generation_results.csv",
    index=False,
)

TEST_GENERATION_SUMMARY.to_csv(
    FINAL_OUTPUT_DIR / "test_generation_summary.csv",
    index=False,
)

display(TEST_GENERATION_SUMMARY)

print("\nDEV automatic-quality reference: 0.7173")
print(
    "TEST automatic-quality score    :",
    round(
        float(
            TEST_GENERATION_SUMMARY[
                "automatic_quality_score"
            ].iloc[0]
        ),
        4,
    ),
)

print("\n🔒 Do not retune the frozen v1 stack after seeing TEST.")


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

,questions,semantic_correctness,gold_coverage,groundedness,citation_presence,citation_validity,citation_support,language_compliance,latency_seconds,tokens_per_second,peak_vram_gb,automatic_quality_score
0,12,0.669555,0.666367,0.650733,1.0,1.0,0.742818,1.0,7.784247,8.572914,1.537403,0.722046



DEV automatic-quality reference: 0.7173
TEST automatic-quality score    : 0.722

🔒 Do not retune the frozen v1 stack after seeing TEST.


In [10]:
# ============================================================
# CELL 10 — FOUR NO-ANSWER QUESTIONS: MANUAL REVIEW
# ============================================================

review_rows = []

for row in FINAL_NOANSWER_GENERATIONS:
    answer = str(row["answer"]).strip()
    citations = CITATION_RE.findall(answer)

    review_rows.append({
        "question_id": row["question_id"],
        "language": row.get("language"),
        "question": row["question"],
        "answer": answer,
        "citation_count": len(citations),
        "refused_correctly": None,
        "false_answer": None,
        "unsupported_citation": None,
        "language_policy_issue": None,
        "manual_notes": "",
    })

NOANSWER_REVIEW = pd.DataFrame(review_rows)

NOANSWER_REVIEW.to_csv(
    FINAL_OUTPUT_DIR / "noanswer_evaluation.csv",
    index=False,
)

for _, row in NOANSWER_REVIEW.iterrows():
    print("\n" + "=" * 100)
    print("ID      :", row["question_id"])
    print("Language:", row["language"])
    print("QUESTION:")
    print(row["question"])
    print("\nQWEN ANSWER:")
    print(row["answer"])
    print("Citations:", row["citation_count"])

print(
    "\n✅ Saved:",
    FINAL_OUTPUT_DIR / "noanswer_evaluation.csv",
)



ID      : q059
Language: en
QUESTION:
What was the exact electricity cost of training the transformer model described in these documents?

QWEN ANSWER:
Not found in the provided documents.
Citations: 0

ID      : q060
Language: fr
QUESTION:
Quel est le numéro de téléphone personnel de l'enseignant du cours de régression ?

QWEN ANSWER:
Not found in the provided documents.
Citations: 0

ID      : q061
Language: en
QUESTION:
What GPU model was used to train the examples in the RNN lecture?

QWEN ANSWER:
Not found in the provided documents.
Citations: 0

ID      : q062
Language: fr
QUESTION:
Quel est le salaire annuel de l'auteur du cours SVM ?

QWEN ANSWER:
Not found in the provided documents.
Citations: 0

✅ Saved: /kaggle/working/docmind_final_v1/noanswer_evaluation.csv


In [11]:
# ============================================================
# CELL 11 — FINAL ARTIFACT CHECK
# ============================================================

required_outputs = [
    "qwen3_4b__test_generations.json",
    "qwen3_4b__noanswer_generations.json",
    "test_generation_results.csv",
    "test_generation_summary.csv",
    "noanswer_evaluation.csv",
]

print("=" * 90)
print("DOCMIND RESEARCH V1 — FINAL GENERATION ARTIFACTS")
print("=" * 90)

for p in sorted(FINAL_OUTPUT_DIR.iterdir()):
    if p.is_file():
        print(
            f"{p.name:<48}"
            f"{p.stat().st_size / 1024:>10.1f} KB"
        )

missing = [
    name
    for name in required_outputs
    if not (FINAL_OUTPUT_DIR / name).exists()
]

if missing:
    print("\n❌ Missing:", missing)
else:
    print("\n🎉 ALL REQUIRED FINAL GENERATION FILES EXIST.")
    print("Save /kaggle/working/docmind_final_v1 as a Kaggle output/dataset.")


DOCMIND RESEARCH V1 — FINAL GENERATION ARTIFACTS
noanswer_evaluation.csv                                0.6 KB
qwen3_4b__noanswer_generations.json                    1.8 KB
qwen3_4b__test_generations.json                        9.6 KB
test_generation_results.csv                            2.3 KB
test_generation_runtime_summary.csv                    0.2 KB
test_generation_summary.csv                            0.4 KB

🎉 ALL REQUIRED FINAL GENERATION FILES EXIST.
Save /kaggle/working/docmind_final_v1 as a Kaggle output/dataset.


In [13]:
import shutil

# Compress /kaggle/working/docmind_research into docmind_research.zip
shutil.make_archive('docmind_final_v11', 'zip', '/kaggle/working/docmind_final_v1')

'/kaggle/working/docmind_final_v11.zip'